# Customise the system model

OpenMM-based `Protocol`s allow you to customise a number of settings which control how the system is modelled. This includes the water model, solvent box shape and padding. Because the protocols share a consistent design, the guide bellow applies to:


- `AbsoluteBindingProtocol`
- `AbsoluteSolvationProtocol`
- `RelativeHybridTopologyProtocol`
- `SepTopProtocol`


This guide covers:

1. [Water model](#Change-the-water-model)
2. [Solvent box shape and padding](#Change-the-solvent-box-shape-and-padding)

The examples here use the `RelativeHybridTopologyProtocol`. The same settings exist for the other protocols, but the names of the groups differ when a protocol has more than one leg (e.g. `solvent_solvation_settings` and `complex_solvation_settings` for `AbsoluteBindingProtocol` and `SepTopProtocol`; `AbsoluteSolvationProtocol` uses a single `solvation_settings`). Check the settings of your protocol with `default_settings()`, and apply each change to every relevant leg.

For more information on these settings see the [User Guide](https://docs.openfree.energy/en/stable/guide/index.html).

In [2]:
from openff.units import unit
from openfe.protocols.openmm_rfe import RelativeHybridTopologyProtocol

settings = RelativeHybridTopologyProtocol.default_settings()

## Changing the water model

Changing the water model for a system with a `SolventComponent` (a system to be solvated at run time) involves changing two coupled settings:

- `solvation_settings.solvent_model`: the model OpenMM uses to solvate the system and generate the initial configuration.
- `forcefield_settings.forcefields`: the force fields used to parameterise the system. The water model should have a compatible structure to the selected force field.

For systems with an `ExplicitSolventComponent` (a system that is already solvated) or for crystal waters in a `ProteinComponent`, the force field determines the water model. It also adds any missing particles from the system, such as hydrogen or virtual sites, to match the force field description.

The default water model is `tip3p`. The supported options are `tip3p`, `spce`, `tip4pew` and `tip5p`, so 4-point (`tip4pew`) and 5-point (`tip5p`) models are available. See the [OpenMM documentation](https://docs.openmm.org/latest/userguide/application/03_model_building_editing.html#adding-solvent) for details on solvation.

Four- and five-point models add virtual site particles, which are handled when the system is solvated.

In [4]:
# Update the water model to a 4-point model
settings.solvation_settings.solvent_model = "tip4pew"
# Update the force field files to match the water model - note the protein force field is also set here and must be kept
settings.forcefield_settings.forcefields = [
    "amber/ff14SB.xml",
    "amber/phosaa10.xml",
    "amber/lipid17_merged.xml",
    "amber/tip4pew_standard.xml",
]

**Things to consider**

* Make sure the water model is compatible with the chosen protein and small-molecule force fields.
* Changing `solvent_model` only changes the water used to solvate. Check that the water and ion parameter files in `forcefield_settings.forcefields` match the chosen model.
* Models with virtual sites may take longer to simulate.

## Changing the solvent box shape and padding

The size and shape of the simulation cell is controlled by `solvation_settings`. The default is a `dodecahedron` with `solvent_padding` of `1.5 nanometers` (the relative hybrid topology protocol reduces this to `1 nanometer` when the system contains a protein). The box shape can be `cube`, `dodecahedron` or `octahedron`.

<div class="alert alert-warning">

**Note:** The simulation cell shape and solvent padding only apply when the system is solvated at run time. If the system is already solvated, these settings are ignored.

</div>

In [5]:
# Change the box shape and padding
settings.solvation_settings.box_shape = "cube"
# Use units from openff.units to specify the padding
settings.solvation_settings.solvent_padding = 1.2 * unit.nanometer

**Things to consider**

* `solvent_padding` is the minimum distance from the solute to the cell edge. For the same padding, a cube contains more water, and so is **slower to simulate**, than a dodecahedron or octahedron.
* Keep the padding large enough to stop the solute interact with its own periodic image, and larger than the nonbonded cutoff. If the padding is too small OpenMM will throw an error during the simulation.
* The box can alternatively be defined with `number_of_solvent_molecules` (with `box_shape`), `box_size` or `box_vectors`. These cannot be combined with `solvent_padding`, so set `solvent_padding = None` first.

In [6]:
# Alternative: define the box by the number of solvent molecules
settings.solvation_settings.solvent_padding = None
settings.solvation_settings.number_of_solvent_molecules = 3000

## Create the Protocol

In [8]:
protocol = RelativeHybridTopologyProtocol(settings=settings)